# BMW i3 Real-World Driving Dataset Pipeline

**MSc Data Science Dissertation**

**Author:** Tarek Noman

**University:** University of East London (UEL)

**Dissertation Title:**
Machine Learning-Based Prediction of Real-World Electric Vehicle Energy Consumption Using Multi-Source Driving Data

---

## Version History

**Version:** 1.0

**Created:** July 2026

### Purpose

This notebook implements the complete preprocessing pipeline for the BMW i3 real-world driving dataset used in this dissertation.

The pipeline is fully reproducible and documents all preprocessing decisions, data validation procedures, feature engineering steps, and target construction methods.

## Stage 1 — Imports

### Objective

Import the Python libraries required for file management, tabular data processing, numerical analysis, and reproducible dataset validation.

In [1]:
from pathlib import Path
import os
import glob

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("Python environment ready.")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Python environment ready.
Pandas version: 3.0.3
NumPy version: 2.4.6


## Stage 2 — Dataset Paths

### Objective

Define all raw-data and processed-data paths in one place. Centralising paths makes the pipeline easier to reproduce and update.

In [2]:
PROJECT_ROOT = Path(r"C:\Projects\DS7010_Dissertation")

RAW_DATA_DIR = (
    PROJECT_ROOT
    / "01-Datasets"
    / "02-BMW-i3"
    / "archive"
)

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "02-Processed"
    / "BMW"
)

OVERVIEW_PATH = RAW_DATA_DIR / "Overview.xlsx"
README_PATH = RAW_DATA_DIR / "readin.m"

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

csv_files = sorted(
    RAW_DATA_DIR.glob("Trip*.csv")
)

print("Project root:", PROJECT_ROOT)
print("Raw BMW folder exists:", RAW_DATA_DIR.exists())
print("Overview file exists:", OVERVIEW_PATH.exists())
print("CSV trip files found:", len(csv_files))
print("Processed-data folder:", PROCESSED_DATA_DIR)


Project root: C:\Projects\DS7010_Dissertation
Raw BMW folder exists: True
Overview file exists: True
CSV trip files found: 70
Processed-data folder: C:\Projects\DS7010_Dissertation\02-Processed\BMW


## Stage 3 — Load Metadata

### Objective

Load the official trip-level metadata from `Overview.xlsx`, remove blank spreadsheet rows and separator columns, and create a variable identifying the TripA and TripB collection groups.

In [3]:
overview = pd.read_excel(
    OVERVIEW_PATH,
    sheet_name="Overview",
    header=0
)

# Remove completely blank rows
overview = overview.dropna(
    how="all"
).copy()

# Keep only valid TripA and TripB records
overview = overview[
    overview["Trip"]
    .astype(str)
    .str.match(r"^Trip[AB]\d+$")
].copy()

# Remove blank separator columns
unnamed_metadata_cols = [
    col for col in overview.columns
    if str(col).startswith("Unnamed")
]

overview = overview.drop(
    columns=unnamed_metadata_cols,
    errors="ignore"
)

# Create trip group
overview["Trip_Group"] = (
    overview["Trip"]
    .astype(str)
    .str.extract(r"^(Trip[AB])")[0]
)

print("Metadata shape:", overview.shape)
print("Unique trips:", overview["Trip"].nunique())

print("\nTrip-group counts:")
print(overview["Trip_Group"].value_counts())

display(overview.head())

Metadata shape: (70, 15)
Unique trips: 70

Trip-group counts:
Trip_Group
TripB    38
TripA    32
Name: count, dtype: int64


,Trip,Date,Route/Area,Weather,Battery Temperature (Start) [°C],Battery Temperature (End),Battery State of Charge (Start),Battery State of Charge (End),Ambient Temperature (Start) [°C],Target Cabin Temperature,Distance [km],Duration [min],Fan,Note,Trip_Group
0,TripA01,2019-06-25_13-21-14,Munich East,sunny,21.0,22.0,0.863,0.803,25.5,23.0,7.427690,16.820000,"Automatic, Level 1",NaN,TripA
1,TripA02,2019-06-25_14-05-31,Munich East,sunny,23.0,26.0,0.803,0.673,32.0,23.0,23.509709,23.550000,"Automatic, Level 1",Target Cabin Temperature changed,TripA
2,TripA03,2019-06-28_10-02-15,Munich East,sunny,24.0,25.0,0.835,0.751,21.5,27.0,12.820846,11.180000,"Automatic, Level 1",Target Cabin Temperature changed,TripA
3,TripA04,2019-06-28_10-13-30,Munich East,sunny,25.0,27.0,0.751,0.667,24.0,22.0,10.727491,6.870000,"Automatic, Level 1",NaN,TripA
4,TripA05,2019-06-28_10-20-26,Munich East,sunny,27.0,27.0,0.667,0.602,24.5,24.0,12.393223,22.776667,"Automatic, Level 1",NaN,TripA


## Stage 4 — Schema Validation

### Objective

Audit every raw trip file before combining them. This stage identifies differences in variable availability, malformed headers, and unexpected trailing columns while preserving the original files unchanged.

In [4]:
schema_records = []

for file_path in csv_files:
    trip_id = file_path.stem

    temp = pd.read_csv(
        file_path,
        sep=";",
        encoding="latin1",
        nrows=5
    )

    schema_records.append({
        "Trip_ID": trip_id,
        "Trip_Group": trip_id[:5],
        "Rows_Read": len(temp),
        "Number_of_Columns": len(temp.columns),
        "Column_List": tuple(temp.columns)
    })

schema_audit = pd.DataFrame(schema_records)

print("Files audited:", len(schema_audit))

print("\nColumn counts by trip group:")
print(
    schema_audit.groupby(
        ["Trip_Group", "Number_of_Columns"]
    ).size()
)

print("\nUnique schemas by trip group:")
print(
    schema_audit.groupby(
        "Trip_Group"
    )["Column_List"].nunique()
)

Files audited: 70

Column counts by trip group:
Trip_Group  Number_of_Columns
TripA       22                    1
            23                   18
            24                    1
            28                   12
TripB       48                   38
dtype: int64

Unique schemas by trip group:
Trip_Group
TripA    4
TripB    2
Name: Column_List, dtype: int64


In [5]:
# Identify variable availability across all files
all_trip_columns = {}

for file_path in csv_files:
    temp = pd.read_csv(
        file_path,
        sep=";",
        encoding="latin1",
        nrows=5
    )

    all_trip_columns[file_path.stem] = set(
        temp.columns
    )

column_presence = {}

for columns in all_trip_columns.values():
    for column in columns:
        column_presence[column] = (
            column_presence.get(column, 0) + 1
        )

column_presence_df = (
    pd.DataFrame(
        column_presence.items(),
        columns=[
            "Variable",
            "Files_Present"
        ]
    )
    .sort_values(
        ["Files_Present", "Variable"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print("Variables present in all 70 files:")
display(
    column_presence_df[
        column_presence_df["Files_Present"] == 70
    ]
)

print("\nKnown malformed columns:")
for trip_id, columns in all_trip_columns.items():
    if "Velocity [km/h]]]" in columns:
        print(
            trip_id,
            "contains malformed velocity header"
        )

    if any(
        str(col).startswith("Unnamed")
        for col in columns
    ):
        print(
            trip_id,
            "contains an unnamed column"
        )

Variables present in all 70 files:


,Variable,Files_Present
0,AirCon Power [kW],70
1,Ambient Temperature [°C],70
2,Battery Current [A],70
3,Battery Temperature [°C],70
4,Battery Voltage [V],70
5,Cabin Temperature Sensor [°C],70
6,Elevation [m],70
7,Heat Exchanger Temperature [°C],70
8,Heater Signal,70
9,Heating Power CAN [kW],70



Known malformed columns:
TripA11 contains an unnamed column
TripB38 contains malformed velocity header


## Stage 5 — Safe BMW Loader

### Objective

Create a reusable loading function that reads each raw trip consistently, repairs the known TripB38 velocity-header error in memory, removes malformed rows containing unexpected trailing values, and adds trip identifiers. The original raw files are not modified.

In [6]:
def load_bmw_trip(
    file_path: Path
) -> pd.DataFrame:
    """
    Load and standardise one BMW i3 trip CSV.

    The raw source file remains unchanged.

    Parameters
    ----------
    file_path:
        Path to one BMW trip CSV file.

    Returns
    -------
    pd.DataFrame
        Cleaned trip-level time-series data with
        Trip_ID and Trip_Group columns.
    """

    trip_id = file_path.stem

    df = pd.read_csv(
        file_path,
        sep=";",
        encoding="latin1"
    )

    # Correct the known TripB38 header typo
    df = df.rename(
        columns={
            "Velocity [km/h]]]":
            "Velocity [km/h]"
        }
    )

    # Identify unexpected trailing columns
    unnamed_cols = [
        col for col in df.columns
        if str(col).startswith("Unnamed")
    ]

    # Remove only rows containing malformed
    # trailing values
    for col in unnamed_cols:
        malformed_mask = df[col].notna()

        if malformed_mask.any():
            df = df.loc[
                ~malformed_mask
            ].copy()

    # Remove the trailing unnamed columns
    df = df.drop(
        columns=unnamed_cols,
        errors="ignore"
    )

    # Add trip identifiers
    df["Trip_ID"] = trip_id
    df["Trip_Group"] = trip_id[:5]

    return df

In [7]:
test_a11 = load_bmw_trip(
    RAW_DATA_DIR / "TripA11.csv"
)

test_b38 = load_bmw_trip(
    RAW_DATA_DIR / "TripB38.csv"
)

print(
    "TripA11 cleaned shape:",
    test_a11.shape
)

print(
    "TripB38 cleaned shape:",
    test_b38.shape
)

print(
    "TripB38 standard velocity header:",
    "Velocity [km/h]"
    in test_b38.columns
)

print(
    "Remaining unnamed columns in TripA11:",
    [
        col for col in test_a11.columns
        if str(col).startswith("Unnamed")
    ]
)

TripA11 cleaned shape: (14244, 25)
TripB38 cleaned shape: (16429, 50)
TripB38 standard velocity header: True
Remaining unnamed columns in TripA11: []


## Stage 6 — Load All Trips

### Objective

Load all 70 trips through the validated loading function and combine them into one analysis dataframe. Optional thermal variables remain missing where they were not collected; no false zeros are introduced.

In [8]:
bmw_trip_frames = []
loading_log = []

for file_path in csv_files:
    try:
        trip_df = load_bmw_trip(
            file_path
        )

        bmw_trip_frames.append(
            trip_df
        )

        loading_log.append({
            "Trip_ID": file_path.stem,
            "Status": "Loaded",
            "Rows": len(trip_df),
            "Columns": len(trip_df.columns)
        })

    except Exception as error:
        loading_log.append({
            "Trip_ID": file_path.stem,
            "Status": "Failed",
            "Rows": np.nan,
            "Columns": np.nan,
            "Error": str(error)
        })

loading_log_df = pd.DataFrame(
    loading_log
)

print("Loading results:")
print(
    loading_log_df["Status"]
    .value_counts(dropna=False)
)

failed_files = loading_log_df[
    loading_log_df["Status"] == "Failed"
]

if not failed_files.empty:
    display(failed_files)

bmw_all = pd.concat(
    bmw_trip_frames,
    ignore_index=True,
    sort=False
)

print("\nCombined BMW shape:", bmw_all.shape)
print(
    "Unique trips:",
    bmw_all["Trip_ID"].nunique()
)

print("\nRows by trip group:")
print(
    bmw_all["Trip_Group"]
    .value_counts()
)

print(
    "\nTotal combined columns:",
    len(bmw_all.columns)
)

Loading results:
Status
Loaded    70
Name: count, dtype: int64

Combined BMW shape: (1094792, 50)
Unique trips: 70

Rows by trip group:
Trip_Group
TripB    627092
TripA    467700
Name: count, dtype: int64

Total combined columns: 50


In [9]:
required_columns = [
    "Trip_ID",
    "Trip_Group",
    "Time [s]",
    "Velocity [km/h]",
    "Battery Voltage [V]",
    "Battery Current [A]",
    "SoC [%]",
    "Ambient Temperature [°C]",
    "Heating Power CAN [kW]",
    "AirCon Power [kW]"
]

missing_required_columns = [
    column
    for column in required_columns
    if column not in bmw_all.columns
]

if missing_required_columns:
    raise ValueError(
        "Missing required columns: "
        + str(missing_required_columns)
    )

print("All required columns are present.")

print("\nMissing values in key variables:")
print(
    bmw_all[
        required_columns
    ]
    .isna()
    .sum()
)

All required columns are present.

Missing values in key variables:
Trip_ID                         0
Trip_Group                      0
Time [s]                        0
Velocity [km/h]                 0
Battery Voltage [V]             0
Battery Current [A]             0
SoC [%]                     30793
Ambient Temperature [°C]        0
Heating Power CAN [kW]          0
AirCon Power [kW]               0
dtype: int64


## Stage 7 — Temporal Audit

### Objective

Verify trip ordering, sampling intervals, duplicate timestamps, and temporal continuity before calculating battery energy and distance. All calculations are performed within individual trips to prevent accidental integration across trip boundaries.


In [10]:
# Sort observations chronologically within each trip
bmw_all = (
    bmw_all
    .sort_values(["Trip_ID", "Time [s]"])
    .reset_index(drop=True)
)

# Calculate elapsed time between consecutive records
bmw_all["Delta_t_s"] = (
    bmw_all
    .groupby("Trip_ID")["Time [s]"]
    .diff()
)

print("Time interval statistics:")
print(bmw_all["Delta_t_s"].describe())

print("\nMost common time intervals:")
print(
    bmw_all["Delta_t_s"]
    .value_counts()
    .head(15)
)

print("\nIntervals <= 0 seconds:")
print((bmw_all["Delta_t_s"] <= 0).sum())

print("\nIntervals > 1 second:")
print((bmw_all["Delta_t_s"] > 1).sum())

print("\nIntervals > 5 seconds:")
print((bmw_all["Delta_t_s"] > 5).sum())

print("\nIntervals > 30 seconds:")
print((bmw_all["Delta_t_s"] > 30).sum())

Time interval statistics:
count    1.094722e+06
mean     1.000001e-01
std      9.557583e-05
min      1.000000e-01
25%      1.000000e-01
50%      1.000000e-01
75%      1.000000e-01
max      2.000000e-01
Name: Delta_t_s, dtype: float64

Most common time intervals:
Delta_t_s
0.1    370789
0.1    333061
0.1    121545
0.1    103040
0.1     70651
0.1     31341
0.1     25760
0.1     17920
0.1      6440
0.1      6056
0.1      4480
0.1      1610
0.1      1120
0.1       419
0.1       280
Name: count, dtype: int64

Intervals <= 0 seconds:
0

Intervals > 1 second:
0

Intervals > 5 seconds:
0

Intervals > 30 seconds:
0


In [11]:
duplicate_timestamp_rows = (
    bmw_all
    .duplicated(
        subset=["Trip_ID", "Time [s]"],
        keep=False
    )
)

print(
    "Rows involved in duplicate trip timestamps:",
    duplicate_timestamp_rows.sum()
)

if duplicate_timestamp_rows.any():
    display(
        bmw_all.loc[
            duplicate_timestamp_rows,
            ["Trip_ID", "Time [s]"]
        ]
        .sort_values(["Trip_ID", "Time [s]"])
        .head(30)
    )

Rows involved in duplicate trip timestamps: 0


In [12]:
trip_time_audit = (
    bmw_all.groupby("Trip_ID")
    .agg(
        Rows=("Time [s]", "size"),
        Start_Time_s=("Time [s]", "min"),
        End_Time_s=("Time [s]", "max"),
        Maximum_Gap_s=("Delta_t_s", "max"),
        Median_Gap_s=("Delta_t_s", "median")
    )
    .reset_index()
)

trip_time_audit["Recorded_Duration_s"] = (
    trip_time_audit["End_Time_s"]
    - trip_time_audit["Start_Time_s"]
)

print("Trip-level temporal audit:")
display(trip_time_audit.describe())

print("\nTrips with gaps above 5 seconds:")
display(
    trip_time_audit[
        trip_time_audit["Maximum_Gap_s"] > 5
    ]
    .sort_values(
        "Maximum_Gap_s",
        ascending=False
    )
)

Trip-level temporal audit:


,Rows,Start_Time_s,End_Time_s,Maximum_Gap_s,Median_Gap_s,Recorded_Duration_s
count,70.000000,70.0,70.000000,70.000000,7.000000e+01,70.000000
mean,15639.885714,0.0,1563.890000,0.101429,1.000000e-01,1563.890000
std,8878.965496,0.0,887.896322,0.011952,3.076867e-14,887.896322
min,3545.000000,0.0,354.400000,0.100000,1.000000e-01,354.400000
25%,10116.250000,0.0,1011.525000,0.100000,1.000000e-01,1011.525000
50%,14079.500000,0.0,1407.850000,0.100000,1.000000e-01,1407.850000
75%,18313.750000,0.0,1831.275000,0.100000,1.000000e-01,1831.275000
max,56102.000000,0.0,5610.100000,0.200000,1.000000e-01,5610.100000



Trips with gaps above 5 seconds:


,Trip_ID,Rows,Start_Time_s,End_Time_s,Maximum_Gap_s,Median_Gap_s,Recorded_Duration_s


### Battery-Current Sign Convention

The current sign convention must be verified before calculating battery power. The analysis compares battery current across speed regimes and regenerative-braking states rather than assuming that the BMW convention matches VED.

In [13]:
print("Battery current statistics:")
print(
    bmw_all["Battery Current [A]"]
    .describe()
)

bmw_all["Driving_State"] = pd.cut(
    bmw_all["Velocity [km/h]"],
    bins=[-0.01, 1, 30, 70, float("inf")],
    labels=[
        "Stopped",
        "Low speed",
        "Medium speed",
        "High speed"
    ]
)

current_by_speed = (
    bmw_all.groupby(
        "Driving_State",
        observed=True
    )["Battery Current [A]"]
    .agg(
        Rows="size",
        Mean_Current_A="mean",
        Median_Current_A="median",
        Minimum_Current_A="min",
        Maximum_Current_A="max"
    )
)

display(current_by_speed)

Battery current statistics:
count    1.094792e+06
mean    -1.730302e+01
std      4.443508e+01
min     -4.043800e+02
25%     -3.127000e+01
50%     -1.146000e+01
75%     -1.690000e+00
max      1.444900e+02
Name: Battery Current [A], dtype: float64


,Rows,Mean_Current_A,Median_Current_A,Minimum_Current_A,Maximum_Current_A
Driving_State,,,,,
Stopped,172229,14.331906,-2.90,-30.69000,125.67
Low speed,245770,-9.880518,-5.82,-184.12000,48.57
Medium speed,419290,-17.576545,-14.58,-401.34698,123.50
High speed,257503,-45.100750,-35.76,-404.38000,144.49


In [14]:
print("Regenerative braking signal values:")
print(
    bmw_all[
        "Regenerative Braking Signal "
    ]
    .value_counts(dropna=False)
    .head(20)
)

regen_current_audit = (
    bmw_all.groupby(
        "Regenerative Braking Signal ",
        dropna=False
    )["Battery Current [A]"]
    .agg(
        Rows="size",
        Mean_Current_A="mean",
        Median_Current_A="median",
        Minimum_Current_A="min",
        Maximum_Current_A="max"
    )
    .reset_index()
)

display(regen_current_audit)

Regenerative braking signal values:
Regenerative Braking Signal 
0.00    1037330
1.00      56657
0.17         12
0.57         12
0.60         12
0.18         12
0.47         11
0.58         11
0.78         11
0.74         11
0.28         11
0.19         10
0.88         10
0.46         10
0.50         10
0.81          9
0.70          9
0.43          9
0.55          9
0.29          9
Name: count, dtype: int64


,Regenerative Braking Signal,Rows,Mean_Current_A,Median_Current_A,Minimum_Current_A,Maximum_Current_A
0,0.00000,1037330,-20.046571,-12.91000,-404.38000,135.80000
1,0.00100,1,38.936080,38.93608,38.93608,38.93608
2,0.01000,4,20.332500,14.41500,-1.64000,54.14000
3,0.01599,1,8.839170,8.83917,8.83917,8.83917
4,0.01852,1,26.461650,26.46165,26.46165,26.46165
...,...,...,...,...,...,...
249,0.98498,1,18.157120,18.15712,18.15712,18.15712
250,0.99000,6,30.685000,20.32000,-6.94000,75.05000
251,0.99450,1,-13.996680,-13.99668,-13.99668,-13.99668
252,0.99950,1,39.385020,39.38502,39.38502,39.38502


## Stage 8 — Energy Target Construction

### 8.1 Battery Power Calculation

### Objective

Calculate the instantaneous battery power at every observation using battery voltage and battery current measurements.

Battery power is calculated before energy integration because energy is obtained by integrating power over time.

The original sign convention of the BMW dataset is preserved, where positive battery current represents charging (regenerative braking) and negative battery current represents battery discharge.

In [15]:
# Battery power (W)
bmw_all["Battery_Power_W"] = (
    bmw_all["Battery Voltage [V]"] *
    bmw_all["Battery Current [A]"]
)

# Convert to kW
bmw_all["Battery_Power_kW"] = (
    bmw_all["Battery_Power_W"] / 1000
)

print("Battery power statistics:")
print(
    bmw_all["Battery_Power_kW"].describe()
)

print("\nCharging samples:")
print(
    (bmw_all["Battery_Power_kW"] > 0).sum()
)

print("\nDischarging samples:")
print(
    (bmw_all["Battery_Power_kW"] < 0).sum()
)

print("\nZero-power samples:")
print(
    (bmw_all["Battery_Power_kW"] == 0).sum()
)

Battery power statistics:
count    1.094792e+06
mean    -6.225522e+00
std      1.617705e+01
min     -1.404904e+02
25%     -1.169818e+01
50%     -4.336085e+00
75%     -6.440428e-01
max      5.564228e+01
Name: Battery_Power_kW, dtype: float64

Charging samples:
225028

Discharging samples:
869647

Zero-power samples:
117


### 8.2 Energy Integration

### Objective

Convert instantaneous battery power into cumulative battery energy using numerical integration.

Energy is calculated independently for each trip using the trapezoidal integration rule. This approach reduces numerical integration error and avoids integrating across trip boundaries.

The resulting energy represents the net battery energy exchanged during each trip, accounting for both battery discharge and regenerative charging.

In [16]:
# Previous battery power within each trip
bmw_all["Battery_Power_kW_Previous"] = (
    bmw_all
    .groupby("Trip_ID")["Battery_Power_kW"]
    .shift(1)
)

# Average power between consecutive observations
bmw_all["Average_Power_kW"] = (
    bmw_all["Battery_Power_kW"] +
    bmw_all["Battery_Power_kW_Previous"]
) / 2

# Incremental energy (kWh)
bmw_all["Energy_Increment_kWh"] = (
    bmw_all["Average_Power_kW"] *
    bmw_all["Delta_t_s"] /
    3600
)

# First record of every trip has no previous point
bmw_all["Energy_Increment_kWh"] = (
    bmw_all["Energy_Increment_kWh"]
    .fillna(0)
)

print("Energy increment statistics:")
print(
    bmw_all["Energy_Increment_kWh"]
    .describe()
)

print("\nMissing values:")
print(
    bmw_all["Energy_Increment_kWh"]
    .isna().sum()
)

Energy increment statistics:
count    1.094792e+06
mean    -1.729279e-04
std      4.491381e-04
min     -3.901272e-03
25%     -3.248855e-04
50%     -1.203955e-04
75%     -1.774104e-05
max      1.514545e-03
Name: Energy_Increment_kWh, dtype: float64

Missing values:
0


In [17]:
trip_energy = (
    bmw_all
    .groupby("Trip_ID")
    .agg(
        Net_Energy_kWh=(
            "Energy_Increment_kWh",
            "sum"
        ),

        Charging_Energy_kWh=(
            "Energy_Increment_kWh",
            lambda x: x[x > 0].sum()
        ),

        Discharging_Energy_kWh=(
            "Energy_Increment_kWh",
            lambda x: -x[x < 0].sum()
        )
    )
    .reset_index()
)

print("Trip energy statistics:")
display(
    trip_energy.describe()
)

print("\nLowest-energy trips:")
display(
    trip_energy
    .sort_values("Net_Energy_kWh")
    .head(10)
)

print("\nHighest-energy trips:")
display(
    trip_energy
    .sort_values(
        "Net_Energy_kWh",
        ascending=False
    )
    .head(10)
)

Trip energy statistics:


,Net_Energy_kWh,Charging_Energy_kWh,Discharging_Energy_kWh
count,70.000000,70.000000,70.000000
mean,-2.704572,1.089678,3.794250
std,3.126441,3.177094,2.177372
min,-11.260721,0.028328,0.625015
25%,-3.477560,0.347710,2.399164
50%,-2.660526,0.522158,3.442802
75%,-1.774983,0.853801,4.268939
max,15.871137,25.572515,12.441929



Lowest-energy trips:


,Trip_ID,Net_Energy_kWh,Charging_Energy_kWh,Discharging_Energy_kWh
45,TripB14,-11.260721,1.181209,12.441929
41,TripB10,-9.994159,0.585140,10.579299
6,TripA07,-6.550272,0.777800,7.328072
32,TripB01,-6.404801,1.857230,8.262032
67,TripB36,-6.171586,0.887088,7.058674
31,TripA32,-6.145973,1.023140,7.169113
38,TripB07,-5.606923,0.578817,6.185741
39,TripB08,-5.230842,0.617192,5.848034
43,TripB12,-5.129012,0.707304,5.836316
5,TripA06,-5.022637,1.207268,6.229905



Highest-energy trips:


,Trip_ID,Net_Energy_kWh,Charging_Energy_kWh,Discharging_Energy_kWh
40,TripB09,15.871137,25.572515,9.701378
35,TripB04,5.783560,9.582805,3.799244
44,TripB13,-0.446232,0.178784,0.625015
13,TripA14,-0.689483,0.242549,0.932031
12,TripA13,-0.691114,0.196846,0.887959
29,TripA30,-1.068564,0.579894,1.648458
0,TripA01,-1.289181,0.382614,1.671795
22,TripA23,-1.323556,0.340585,1.664141
64,TripB33,-1.344167,0.333491,1.677658
65,TripB34,-1.414931,0.028328,1.443259


In [18]:
print("Trips with positive net energy:")
display(
    trip_energy[
        trip_energy["Net_Energy_kWh"] > 0
    ]
    .sort_values(
        "Net_Energy_kWh",
        ascending=False
    )
    .head(10)
)

print("\nTrips with negative net energy:")
display(
    trip_energy[
        trip_energy["Net_Energy_kWh"] < 0
    ]
    .sort_values(
        "Net_Energy_kWh"
    )
    .head(10)
)

Trips with positive net energy:


,Trip_ID,Net_Energy_kWh,Charging_Energy_kWh,Discharging_Energy_kWh
40,TripB09,15.871137,25.572515,9.701378
35,TripB04,5.783560,9.582805,3.799244



Trips with negative net energy:


,Trip_ID,Net_Energy_kWh,Charging_Energy_kWh,Discharging_Energy_kWh
45,TripB14,-11.260721,1.181209,12.441929
41,TripB10,-9.994159,0.585140,10.579299
6,TripA07,-6.550272,0.777800,7.328072
32,TripB01,-6.404801,1.857230,8.262032
67,TripB36,-6.171586,0.887088,7.058674
31,TripA32,-6.145973,1.023140,7.169113
38,TripB07,-5.606923,0.578817,6.185741
39,TripB08,-5.230842,0.617192,5.848034
43,TripB12,-5.129012,0.707304,5.836316
5,TripA06,-5.022637,1.207268,6.229905


### 8.2.1 Energy Integration Validation

In [19]:
trip_energy = trip_energy.merge(
    overview[
        [
            "Trip",
            "Distance [km]",
            "Duration [min]",
            "Battery State of Charge (Start)",
            "Battery State of Charge (End)"
        ]
    ],
    left_on="Trip_ID",
    right_on="Trip",
    how="left"
)

trip_energy["SOC_Drop_pct"] = (
    trip_energy["Battery State of Charge (Start)"] -
    trip_energy["Battery State of Charge (End)"]
)

trip_energy["Energy_Consumed_kWh"] = (
    -trip_energy["Net_Energy_kWh"]
)

print("Merged trip table:")
display(trip_energy.head())

print("\nSOC drop statistics:")
display(
    trip_energy["SOC_Drop_pct"].describe()
)

Merged trip table:


,Trip_ID,Net_Energy_kWh,Charging_Energy_kWh,Discharging_Energy_kWh,Trip,Distance [km],Duration [min],Battery State of Charge (Start),Battery State of Charge (End),SOC_Drop_pct,Energy_Consumed_kWh
0,TripA01,-1.289181,0.382614,1.671795,TripA01,7.427690,16.820000,0.863,0.803,0.060,1.289181
1,TripA02,-2.934639,1.649346,4.583985,TripA02,23.509709,23.550000,0.803,0.673,0.130,2.934639
2,TripA03,-1.954406,0.454204,2.408610,TripA03,12.820846,11.180000,0.835,0.751,0.084,1.954406
3,TripA04,-1.899911,0.260278,2.160189,TripA04,10.727491,6.870000,0.751,0.667,0.084,1.899911
4,TripA05,-1.504167,0.675157,2.179324,TripA05,12.393223,22.776667,0.667,0.602,0.065,1.504167



SOC drop statistics:


count    70.000000
mean      0.129314
std       0.098627
min      -0.241000
25%       0.079000
50%       0.117500
75%       0.157500
max       0.509000
Name: SOC_Drop_pct, dtype: float64

### 8.2.2 Energy–SOC Validation

In [20]:
correlation = trip_energy[
    ["Energy_Consumed_kWh", "SOC_Drop_pct"]
].corr()

print("Correlation matrix:")
display(correlation)

print("\nTrips where SOC increased:")
display(
    trip_energy[
        trip_energy["SOC_Drop_pct"] < 0
    ][
        [
            "Trip_ID",
            "Energy_Consumed_kWh",
            "SOC_Drop_pct",
            "Distance [km]"
        ]
    ]
)

Correlation matrix:


,Energy_Consumed_kWh,SOC_Drop_pct
Energy_Consumed_kWh,1.000000,0.576238
SOC_Drop_pct,0.576238,1.000000



Trips where SOC increased:


,Trip_ID,Energy_Consumed_kWh,SOC_Drop_pct,Distance [km]
30,TripA31,2.113023,-0.015,15.976052
35,TripB04,-5.783560,-0.241,16.595300


### 8.3 Distance Calculation

### Objective

Validate the official trip distances provided in the BMW metadata against distances calculated from the recorded vehicle speed measurements.

Agreement between the two measures provides additional evidence that the target variable is constructed from physically consistent data.

In [21]:
# Convert speed to m/s
bmw_all["Speed_mps"] = (
    bmw_all["Velocity [km/h]"] / 3.6
)

# Distance travelled during each sample (metres)
bmw_all["Distance_Increment_m"] = (
    bmw_all["Speed_mps"] *
    bmw_all["Delta_t_s"]
)

bmw_all["Distance_Increment_m"] = (
    bmw_all["Distance_Increment_m"]
    .fillna(0)
)

trip_distance = (
    bmw_all
    .groupby("Trip_ID")
    .agg(
        Calculated_Distance_m=(
            "Distance_Increment_m",
            "sum"
        )
    )
    .reset_index()
)

trip_distance["Calculated_Distance_km"] = (
    trip_distance["Calculated_Distance_m"] / 1000
)

display(trip_distance.head())

,Trip_ID,Calculated_Distance_m,Calculated_Distance_km
0,TripA01,7427.969444,7.427969
1,TripA02,23511.695833,23.511696
2,TripA03,12820.846111,12.820846
3,TripA04,10727.602500,10.727602
4,TripA05,12393.423333,12.393423


In [22]:
trip_energy = trip_energy.merge(
    trip_distance[
        ["Trip_ID", "Calculated_Distance_km"]
    ],
    on="Trip_ID",
    how="left"
)

trip_energy["Distance_Error_pct"] = (
    (
        trip_energy["Calculated_Distance_km"]
        - trip_energy["Distance [km]"]
    )
    / trip_energy["Distance [km]"]
) * 100

print("Distance comparison statistics:")
display(
    trip_energy[
        [
            "Distance [km]",
            "Calculated_Distance_km",
            "Distance_Error_pct"
        ]
    ].describe()
)

print("\nMedian absolute percentage error:")
print(
    trip_energy["Distance_Error_pct"]
    .abs()
    .median()
)

print("\nLargest distance differences:")
display(
    trip_energy.reindex(
        trip_energy["Distance_Error_pct"]
        .abs()
        .sort_values(ascending=False)
        .index
    ).head(10)
)

Distance comparison statistics:


,Distance [km],Calculated_Distance_km,Distance_Error_pct
count,70.000000,70.000000,7.000000e+01
mean,19.354558,19.397196,5.504808e-01
std,11.615868,11.578085,4.598061e+00
min,2.782300,2.783329,-6.724970e-03
25%,12.500129,12.500279,-3.456615e-13
50%,16.502600,16.502051,3.908091e-13
75%,19.340116,19.340130,2.124960e-04
max,60.974700,60.974758,3.847102e+01



Median absolute percentage error:
6.984378768665244e-05

Largest distance differences:


,Trip_ID,Net_Energy_kWh,Charging_Energy_kWh,Discharging_Energy_kWh,Trip,Distance [km],Duration [min],Battery State of Charge (Start),Battery State of Charge (End),SOC_Drop_pct,Energy_Consumed_kWh,Calculated_Distance_km,Distance_Error_pct
29,TripA30,-1.068564,0.579894,1.648458,TripA30,7.746444,23.288333,0.855,0.810,0.045,1.068564,10.726581,38.471019
44,TripB13,-0.446232,0.178784,0.625015,TripB13,2.782300,5.906667,0.570,0.551,0.019,0.446232,2.783329,0.036980
65,TripB34,-1.414931,0.028328,1.443259,TripB34,9.143200,12.228333,0.739,0.713,0.026,1.414931,9.144609,0.015409
1,TripA02,-2.934639,1.649346,4.583985,TripA02,23.509709,23.550000,0.803,0.673,0.130,2.934639,23.511696,0.008450
37,TripB06,-3.207577,0.335810,3.543388,TripB06,16.580000,22.533333,0.832,0.693,0.139,3.207577,16.578885,-0.006725
35,TripB04,5.783560,9.582805,3.799244,TripB04,16.595300,49.248333,0.451,0.692,-0.241,-5.783560,16.594296,-0.006053
0,TripA01,-1.289181,0.382614,1.671795,TripA01,7.427690,16.820000,0.863,0.803,0.060,1.289181,7.427969,0.003766
28,TripA29,-1.779866,0.939727,2.719593,TripA29,16.332609,22.315000,0.709,0.631,0.078,1.779866,16.333097,0.002985
36,TripB05,-2.811229,0.347449,3.158678,TripB05,14.820000,16.990000,0.719,0.595,0.124,2.811229,14.820263,0.001773
4,TripA05,-1.504167,0.675157,2.179324,TripA05,12.393223,22.776667,0.667,0.602,0.065,1.504167,12.393423,0.001613


### 8.4 Target Construction (kWh/100 km)

### Objective

Construct the trip-level target variable representing energy consumption in kilowatt-hours per 100 kilometres (kWh/100 km).

Energy consumption is calculated using the integrated battery energy and the validated trip distance.

In [23]:
trip_energy["Energy_Consumption_kWh_100km"] = (
    trip_energy["Energy_Consumed_kWh"]
    /
    trip_energy["Calculated_Distance_km"]
) * 100

print("Target statistics:")
display(
    trip_energy["Energy_Consumption_kWh_100km"]
    .describe()
)

print("\nHighest 10 target values:")
display(
    trip_energy.sort_values(
        "Energy_Consumption_kWh_100km",
        ascending=False
    )[
        [
            "Trip_ID",
            "Energy_Consumed_kWh",
            "Calculated_Distance_km",
            "Energy_Consumption_kWh_100km"
        ]
    ].head(10)
)

print("\nLowest 10 target values:")
display(
    trip_energy.sort_values(
        "Energy_Consumption_kWh_100km"
    )[
        [
            "Trip_ID",
            "Energy_Consumed_kWh",
            "Calculated_Distance_km",
            "Energy_Consumption_kWh_100km"
        ]
    ].head(10)
)

Target statistics:


count    70.000000
mean     15.086996
std       8.843588
min     -34.852702
25%      13.279140
50%      16.371685
75%      19.093422
max      23.301433
Name: Energy_Consumption_kWh_100km, dtype: float64


Highest 10 target values:


,Trip_ID,Energy_Consumed_kWh,Calculated_Distance_km,Energy_Consumption_kWh_100km
33,TripB02,4.410032,18.926011,23.301433
54,TripB23,4.074655,18.705172,21.783572
63,TripB32,3.095883,14.240884,21.739404
60,TripB29,3.360785,15.833409,21.225910
61,TripB30,3.136316,14.860657,21.104824
49,TripB18,3.329185,15.813621,21.052643
62,TripB31,3.203053,15.241381,21.015502
68,TripB37,3.667533,17.462634,21.002172
46,TripB15,4.026008,19.239285,20.925972
41,TripB10,9.994159,47.839096,20.891196



Lowest 10 target values:


,Trip_ID,Energy_Consumed_kWh,Calculated_Distance_km,Energy_Consumption_kWh_100km
35,TripB04,-5.783560,16.594296,-34.852702
40,TripB09,-15.871137,54.182571,-29.291961
25,TripA26,1.726815,18.032233,9.576267
14,TripA15,2.665912,26.973717,9.883370
29,TripA30,1.068564,10.726581,9.961835
28,TripA29,1.779866,16.333097,10.897297
5,TripA06,5.022637,45.993461,10.920328
18,TripA19,1.920658,16.730382,11.480062
27,TripA28,2.320283,19.515316,11.889547
4,TripA05,1.504167,12.393423,12.136815


In [24]:
print("Trips below 5 kWh/100 km:")
print(
    (
        trip_energy["Energy_Consumption_kWh_100km"] < 5
    ).sum()
)

print("\nTrips above 40 kWh/100 km:")
print(
    (
        trip_energy["Energy_Consumption_kWh_100km"] > 40
    ).sum()
)

print("\nTrips above 60 kWh/100 km:")
print(
    (
        trip_energy["Energy_Consumption_kWh_100km"] > 60
    ).sum()
)

Trips below 5 kWh/100 km:
2

Trips above 40 kWh/100 km:
0

Trips above 60 kWh/100 km:
0


In [25]:
trip_energy[
    trip_energy["Energy_Consumption_kWh_100km"] < 5
][
[
"Trip_ID",
"Energy_Consumed_kWh",
"Charging_Energy_kWh",
"Discharging_Energy_kWh",
"Calculated_Distance_km",
"SOC_Drop_pct",
"Energy_Consumption_kWh_100km"
]
]

,Trip_ID,Energy_Consumed_kWh,Charging_Energy_kWh,Discharging_Energy_kWh,Calculated_Distance_km,SOC_Drop_pct,Energy_Consumption_kWh_100km
35,TripB04,-5.783560,9.582805,3.799244,16.594296,-0.241,-34.852702
40,TripB09,-15.871137,25.572515,9.701378,54.182571,0.240,-29.291961


### 8.5 Target Validation

### Objective

Evaluate the statistical properties of the constructed energy-consumption target before feature engineering and machine learning.

This stage identifies unrealistic observations, examines the target distribution, and documents any exceptional trips that require further investigation.

In [26]:
print("Target summary statistics:")
display(
    trip_energy["Energy_Consumption_kWh_100km"]
    .describe()
)

print("\nTarget percentiles:")
display(
    trip_energy["Energy_Consumption_kWh_100km"]
    .quantile([
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ])
)

Target summary statistics:


count    70.000000
mean     15.086996
std       8.843588
min     -34.852702
25%      13.279140
50%      16.371685
75%      19.093422
max      23.301433
Name: Energy_Consumption_kWh_100km, dtype: float64


Target percentiles:


0.01   -31.015791
0.05     9.918679
0.10    11.424089
0.25    13.279140
0.50    16.371685
0.75    19.093422
0.90    21.003505
0.95    21.171421
0.99    22.254109
Name: Energy_Consumption_kWh_100km, dtype: float64

In [27]:
Q1 = trip_energy["Energy_Consumption_kWh_100km"].quantile(0.25)
Q3 = trip_energy["Energy_Consumption_kWh_100km"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("Lower fence:", lower)
print("Upper fence:", upper)

outliers = trip_energy[
    (trip_energy["Energy_Consumption_kWh_100km"] < lower) |
    (trip_energy["Energy_Consumption_kWh_100km"] > upper)
]

print("\nNumber of IQR outliers:", len(outliers))

display(
    outliers[
        [
            "Trip_ID",
            "Energy_Consumption_kWh_100km",
            "Calculated_Distance_km",
            "SOC_Drop_pct"
        ]
    ]
)

Lower fence: 4.5577173103104816
Upper fence: 27.814845611190947

Number of IQR outliers: 2


,Trip_ID,Energy_Consumption_kWh_100km,Calculated_Distance_km,SOC_Drop_pct
35,TripB04,-34.852702,16.594296,-0.241
40,TripB09,-29.291961,54.182571,0.240


In [28]:
print("Trips below 0:")
print(
    (
        trip_energy["Energy_Consumption_kWh_100km"] < 0
    ).sum()
)

print("\nTrips below 5:")
print(
    (
        trip_energy["Energy_Consumption_kWh_100km"] < 5
    ).sum()
)

print("\nTrips above 30:")
print(
    (
        trip_energy["Energy_Consumption_kWh_100km"] > 30
    ).sum()
)

print("\nTrips above 40:")
print(
    (
        trip_energy["Energy_Consumption_kWh_100km"] > 40
    ).sum()
)

Trips below 0:
2

Trips below 5:
2

Trips above 30:
0

Trips above 40:
0


In [29]:
print("Correlation with trip distance:")

display(
    trip_energy[
        [
            "Calculated_Distance_km",
            "Energy_Consumption_kWh_100km"
        ]
    ].corr()
)

Correlation with trip distance:


,Calculated_Distance_km,Energy_Consumption_kWh_100km
Calculated_Distance_km,1.00000,-0.21984
Energy_Consumption_kWh_100km,-0.21984,1.00000


### 8.5.1 Distribution Validation

The constructed target variable is evaluated to verify that its distribution is physically realistic before machine learning.

The validation examines:

- Summary statistics
- Percentiles
- Interquartile range (IQR)
- Physical plausibility
- Extreme observations
  

In [30]:
print("="*60)
print("TARGET VALIDATION")
print("="*60)

target = trip_energy["Energy_Consumption_kWh_100km"]

print("\nSummary statistics:")
display(target.describe())

print("\nPercentiles:")
display(
    target.quantile([
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ])
)

print("\nStandard deviation:", target.std())
print("Variance:", target.var())
print("Skewness:", target.skew())
print("Kurtosis:", target.kurt())

TARGET VALIDATION

Summary statistics:


count    70.000000
mean     15.086996
std       8.843588
min     -34.852702
25%      13.279140
50%      16.371685
75%      19.093422
max      23.301433
Name: Energy_Consumption_kWh_100km, dtype: float64


Percentiles:


0.01   -31.015791
0.05     9.918679
0.10    11.424089
0.25    13.279140
0.50    16.371685
0.75    19.093422
0.90    21.003505
0.95    21.171421
0.99    22.254109
Name: Energy_Consumption_kWh_100km, dtype: float64


Standard deviation: 8.843587671543457
Variance: 78.2090429042754
Skewness: -4.504365643960387
Kurtosis: 23.03646954262082


### 8.5.2 Physical Plausibility

Trips with physically unusual energy consumption values are identified for further investigation.

These observations are not removed automatically. Their causes are examined before any modelling decisions are made.

In [31]:
print("Negative consumption trips:")
print((target < 0).sum())

print("\nTrips below 5 kWh/100km:")
print(((target < 5) & (target >= 0)).sum())

print("\nTrips between 5 and 30:")
print(((target >= 5) & (target <= 30)).sum())

print("\nTrips above 30:")
print((target > 30).sum())

print("\nTrips above 40:")
print((target > 40).sum())


Negative consumption trips:
2

Trips below 5 kWh/100km:
0

Trips between 5 and 30:
68

Trips above 30:
0

Trips above 40:
0


### 8.5.3 Statistical Outlier Detection

The interquartile range (IQR) rule is used to identify statistical outliers in the target variable.

This provides an objective method for distinguishing unusual observations from typical driving behaviour.

In [32]:
Q1 = target.quantile(0.25)
Q3 = target.quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)

print("\nLower fence:", lower)
print("Upper fence:", upper)

target_outliers = trip_energy[
    (target < lower) |
    (target > upper)
]

print("\nNumber of outliers:", len(target_outliers))

display(
    target_outliers[
        [
            "Trip_ID",
            "Energy_Consumption_kWh_100km",
            "Calculated_Distance_km",
            "SOC_Drop_pct"
        ]
    ]
)

Q1: 13.279140423140657
Q3: 19.093422498360773
IQR: 5.814282075220117

Lower fence: 4.5577173103104816
Upper fence: 27.814845611190947

Number of outliers: 2


,Trip_ID,Energy_Consumption_kWh_100km,Calculated_Distance_km,SOC_Drop_pct
35,TripB04,-34.852702,16.594296,-0.241
40,TripB09,-29.291961,54.182571,0.240


### Interpretation

The target distribution was inspected using descriptive statistics, physical limits, and the interquartile range method.

The majority of trips fall within the expected energy-consumption range reported for production BMW i3 vehicles under real driving conditions.

A small number of regeneration-dominated trips produce unusually low or negative net energy consumption. These trips are retained in the audit dataset and will be considered separately when constructing the final leakage-safe modelling table.

## Stage 9 — Feature Engineering

## 9.1 Feature Dictionary

### Objective

Construct trip-level predictor variables that summarize driving behaviour, environmental conditions, battery state, and HVAC usage.

These predictors are intended to explain variations in trip energy consumption while avoiding data leakage.

The engineered features are grouped into four categories:

- Driving Behaviour
- Battery State
- Thermal Environment
- Vehicle Usage

In [33]:
# Positive acceleration only
bmw_all["Positive_Acceleration"] = (
    bmw_all["Longitudinal Acceleration [m/s^2]"]
    .clip(lower=0)
)

# Vehicle stopped
bmw_all["Idle"] = (
    bmw_all["Velocity [km/h]"] < 1
).astype(int)

In [34]:
trip_features = (
    bmw_all
    .groupby("Trip_ID")
    .agg(

        # ---------- Driving ----------
        Mean_Speed_kmh=("Velocity [km/h]", "mean"),
        Max_Speed_kmh=("Velocity [km/h]", "max"),
        Std_Speed_kmh=("Velocity [km/h]", "std"),

        Mean_Acceleration_mps2=(
            "Longitudinal Acceleration [m/s^2]",
            "mean"
        ),

        Max_Acceleration_mps2=(
            "Longitudinal Acceleration [m/s^2]",
            "max"
        ),

        Min_Acceleration_mps2=(
            "Longitudinal Acceleration [m/s^2]",
            "min"
        ),

        Mean_Positive_Acceleration_mps2=(
            "Positive_Acceleration",
            "mean"
        ),

        Idling_Proportion=(
            "Idle",
            "mean"
        ),

        # ---------- Battery ----------
        Start_SOC=("SoC [%]", "first"),

        End_SOC=("SoC [%]", "last"),

        # ---------- Thermal ----------
        Mean_Ambient_Temp_C=(
            "Ambient Temperature [°C]",
            "mean"
        ),

        Mean_Battery_Temp_C=(
            "Battery Temperature [°C]",
            "mean"
        ),

        Mean_Heating_kW=(
            "Heating Power CAN [kW]",
            "mean"
        ),

        Mean_AirCon_kW=(
            "AirCon Power [kW]",
            "mean"
        ),

        # ---------- Vehicle ----------
        Duration_s=("Time [s]", "max")

    )
    .reset_index()
)

trip_features["SOC_Drop_pct"] = (
    trip_features["Start_SOC"] -
    trip_features["End_SOC"]
)

display(trip_features.head())

,Trip_ID,Mean_Speed_kmh,Max_Speed_kmh,Std_Speed_kmh,Mean_Acceleration_mps2,Max_Acceleration_mps2,Min_Acceleration_mps2,Mean_Positive_Acceleration_mps2,Idling_Proportion,Start_SOC,End_SOC,Mean_Ambient_Temp_C,Mean_Battery_Temp_C,Mean_Heating_kW,Mean_AirCon_kW,Duration_s,SOC_Drop_pct
0,TripA01,26.502170,53.30,15.497032,0.014246,2.79,-2.29,0.268194,0.125074,86.9,81.5,30.769972,21.940141,0.042883,1.610408,1008.9,5.4
1,TripA02,59.909267,132.70,32.868059,-0.019747,2.50,-3.20,0.277499,0.000000,80.3,67.3,31.127573,24.972960,0.052153,0.456553,1412.9,13.0
2,TripA03,68.826493,143.82,38.047229,-0.020892,3.81,-2.99,0.213431,0.041455,83.5,75.1,23.334916,25.100571,1.453018,0.420975,670.5,8.4
3,TripA04,93.645415,136.94,41.693047,0.062546,3.35,-1.93,0.267561,0.088506,75.1,66.7,24.423957,26.817953,0.317709,0.300182,412.3,8.4
4,TripA05,32.646337,80.83,18.151337,-0.000025,3.31,-3.08,0.249061,0.066364,66.7,60.2,24.587443,27.000000,1.691257,0.304058,1366.6,6.5


In [35]:
bmw_ml = (
    trip_features.merge(
        trip_energy[
            [
                "Trip_ID",
                "Calculated_Distance_km",
                "Energy_Consumption_kWh_100km"
            ]
        ],
        on="Trip_ID",
        how="left"
    )
)

print("ML table shape:", bmw_ml.shape)

display(bmw_ml.head())

ML table shape: (70, 19)


,Trip_ID,Mean_Speed_kmh,Max_Speed_kmh,Std_Speed_kmh,Mean_Acceleration_mps2,Max_Acceleration_mps2,Min_Acceleration_mps2,Mean_Positive_Acceleration_mps2,Idling_Proportion,Start_SOC,End_SOC,Mean_Ambient_Temp_C,Mean_Battery_Temp_C,Mean_Heating_kW,Mean_AirCon_kW,Duration_s,SOC_Drop_pct,Calculated_Distance_km,Energy_Consumption_kWh_100km
0,TripA01,26.502170,53.30,15.497032,0.014246,2.79,-2.29,0.268194,0.125074,86.9,81.5,30.769972,21.940141,0.042883,1.610408,1008.9,5.4,7.427969,17.355765
1,TripA02,59.909267,132.70,32.868059,-0.019747,2.50,-3.20,0.277499,0.000000,80.3,67.3,31.127573,24.972960,0.052153,0.456553,1412.9,13.0,23.511696,12.481613
2,TripA03,68.826493,143.82,38.047229,-0.020892,3.81,-2.99,0.213431,0.041455,83.5,75.1,23.334916,25.100571,1.453018,0.420975,670.5,8.4,12.820846,15.243973
3,TripA04,93.645415,136.94,41.693047,0.062546,3.35,-1.93,0.267561,0.088506,75.1,66.7,24.423957,26.817953,0.317709,0.300182,412.3,8.4,10.727602,17.710491
4,TripA05,32.646337,80.83,18.151337,-0.000025,3.31,-3.08,0.249061,0.066364,66.7,60.2,24.587443,27.000000,1.691257,0.304058,1366.6,6.5,12.393423,12.136815


In [36]:
print("="*60)
print("BMW MACHINE LEARNING TABLE")
print("="*60)

print("\nShape:")
print(bmw_ml.shape)

print("\nMissing values:")
display(
    bmw_ml.isna().sum()
)

print("\nDuplicate trips:")
print(
    bmw_ml["Trip_ID"].duplicated().sum()
)

print("\nColumns:")
print(list(bmw_ml.columns))

BMW MACHINE LEARNING TABLE

Shape:
(70, 19)

Missing values:


Trip_ID                            0
Mean_Speed_kmh                     0
Max_Speed_kmh                      0
Std_Speed_kmh                      0
Mean_Acceleration_mps2             0
Max_Acceleration_mps2              0
Min_Acceleration_mps2              0
Mean_Positive_Acceleration_mps2    0
Idling_Proportion                  0
Start_SOC                          0
End_SOC                            0
Mean_Ambient_Temp_C                0
Mean_Battery_Temp_C                0
Mean_Heating_kW                    0
Mean_AirCon_kW                     0
Duration_s                         0
SOC_Drop_pct                       0
Calculated_Distance_km             0
Energy_Consumption_kWh_100km       0
dtype: int64


Duplicate trips:
0

Columns:
['Trip_ID', 'Mean_Speed_kmh', 'Max_Speed_kmh', 'Std_Speed_kmh', 'Mean_Acceleration_mps2', 'Max_Acceleration_mps2', 'Min_Acceleration_mps2', 'Mean_Positive_Acceleration_mps2', 'Idling_Proportion', 'Start_SOC', 'End_SOC', 'Mean_Ambient_Temp_C', 'Mean_Battery_Temp_C', 'Mean_Heating_kW', 'Mean_AirCon_kW', 'Duration_s', 'SOC_Drop_pct', 'Calculated_Distance_km', 'Energy_Consumption_kWh_100km']


## Stage 10 — Leakage-Safe Modelling Table

### Objective

Construct the final modelling dataset by removing variables that are unavailable at prediction time or that introduce target leakage.

The complete audit dataset is retained separately for transparency and reproducibility.

In [37]:
# Complete audit table
bmw_audit_table = bmw_ml.copy()

# Final modelling table
bmw_ml_final = bmw_ml[
    bmw_ml["Energy_Consumption_kWh_100km"] > 0
].copy()

# Remove leakage and target-derived variables
bmw_ml_final = bmw_ml_final.drop(
    columns=[
        "End_SOC",
        "SOC_Drop_pct",
        "Calculated_Distance_km"
    ]
)

print("=" * 60)
print("FINAL BMW MODELLING TABLE")
print("=" * 60)

print("\nAudit table:", bmw_audit_table.shape)
print("ML table:", bmw_ml_final.shape)

print("\nColumns:")
for col in bmw_ml_final.columns:
    print("-", col)

print("\nMissing values:")
print(bmw_ml_final.isna().sum())

FINAL BMW MODELLING TABLE

Audit table: (70, 19)
ML table: (68, 16)

Columns:
- Trip_ID
- Mean_Speed_kmh
- Max_Speed_kmh
- Std_Speed_kmh
- Mean_Acceleration_mps2
- Max_Acceleration_mps2
- Min_Acceleration_mps2
- Mean_Positive_Acceleration_mps2
- Idling_Proportion
- Start_SOC
- Mean_Ambient_Temp_C
- Mean_Battery_Temp_C
- Mean_Heating_kW
- Mean_AirCon_kW
- Duration_s
- Energy_Consumption_kWh_100km

Missing values:
Trip_ID                            0
Mean_Speed_kmh                     0
Max_Speed_kmh                      0
Std_Speed_kmh                      0
Mean_Acceleration_mps2             0
Max_Acceleration_mps2              0
Min_Acceleration_mps2              0
Mean_Positive_Acceleration_mps2    0
Idling_Proportion                  0
Start_SOC                          0
Mean_Ambient_Temp_C                0
Mean_Battery_Temp_C                0
Mean_Heating_kW                    0
Mean_AirCon_kW                     0
Duration_s                         0
Energy_Consumption_kWh_100km 

## Stage 11 — Export Clean Dataset

## Stage 11 — Export Clean Dataset

### Objective

Export both the complete audit dataset and the leakage-safe modelling dataset for subsequent machine learning experiments.

The audit dataset preserves all observations for transparency and reproducibility, while the modelling dataset contains only validated predictors and the target variable.

In [38]:
# ==========================================================
# Export BMW datasets
# ==========================================================

audit_path = (
    PROCESSED_DATA_DIR
    / "BMW_AUDIT_TABLE.csv"
)

ml_path = (
    PROCESSED_DATA_DIR
    / "BMW_ML_TABLE.csv"
)

bmw_audit_table.to_csv(
    audit_path,
    index=False
)

bmw_ml_final.to_csv(
    ml_path,
    index=False
)

print("BMW Audit Table saved:")
print(audit_path)

print("\nBMW ML Table saved:")
print(ml_path)

print("\nAudit shape:", bmw_audit_table.shape)
print("ML shape:", bmw_ml_final.shape)

BMW Audit Table saved:
C:\Projects\DS7010_Dissertation\02-Processed\BMW\BMW_AUDIT_TABLE.csv

BMW ML Table saved:
C:\Projects\DS7010_Dissertation\02-Processed\BMW\BMW_ML_TABLE.csv

Audit shape: (70, 19)
ML shape: (68, 16)


# BMW battery-energy sign audit

In [39]:
# BMW battery-energy sign audit

print("Battery power sign:")
print("Positive power samples:",
      (bmw_all["Battery_Power_kW"] > 0).sum())
print("Negative power samples:",
      (bmw_all["Battery_Power_kW"] < 0).sum())

print("\nTrip net-energy sign:")
print("Positive net-energy trips:",
      (trip_energy["Net_Energy_kWh"] > 0).sum())
print("Negative net-energy trips:",
      (trip_energy["Net_Energy_kWh"] < 0).sum())
print("Zero net-energy trips:",
      (trip_energy["Net_Energy_kWh"] == 0).sum())

print("\nSOC relationship:")
print(
    trip_energy[
        ["Net_Energy_kWh", "SOC_Drop_pct"]
    ].corr()
)

print("\nTrip energy / SOC examples:")
display(
    trip_energy[
        [
            "Trip_ID",
            "Net_Energy_kWh",
            "SOC_Drop_pct",
            "Battery State of Charge (Start)",
            "Battery State of Charge (End)"
        ]
    ].head(15)
)

Battery power sign:
Positive power samples: 225028
Negative power samples: 869647

Trip net-energy sign:
Positive net-energy trips: 2
Negative net-energy trips: 68
Zero net-energy trips: 0

SOC relationship:
                Net_Energy_kWh  SOC_Drop_pct
Net_Energy_kWh        1.000000     -0.576238
SOC_Drop_pct         -0.576238      1.000000

Trip energy / SOC examples:


,Trip_ID,Net_Energy_kWh,SOC_Drop_pct,Battery State of Charge (Start),Battery State of Charge (End)
0,TripA01,-1.289181,0.060,0.863,0.803
1,TripA02,-2.934639,0.130,0.803,0.673
2,TripA03,-1.954406,0.084,0.835,0.751
3,TripA04,-1.899911,0.084,0.751,0.667
4,TripA05,-1.504167,0.065,0.667,0.602
5,TripA06,-5.022637,0.217,0.851,0.634
6,TripA07,-6.550272,0.292,0.634,0.342
7,TripA08,-2.449564,0.105,0.866,0.761
8,TripA09,-1.432895,0.061,0.747,0.686
9,TripA10,-2.020156,0.087,0.696,0.609


In [40]:
print("Current BMW final shape:", bmw_ml_final.shape)

print("\nTarget sign in current final table:")
print(
    bmw_ml_final["Energy_Consumption_kWh_100km"]
    .describe()
)

print("\nFirst 10 targets:")
display(
    bmw_ml_final[
        ["Trip_ID", "Energy_Consumption_kWh_100km"]
    ].head(10)
)

Current BMW final shape: (68, 16)

Target sign in current final table:
count    68.000000
mean     16.474035
std       3.460082
min       9.576267
25%      13.729263
50%      16.514365
75%      19.115315
max      23.301433
Name: Energy_Consumption_kWh_100km, dtype: float64

First 10 targets:


,Trip_ID,Energy_Consumption_kWh_100km
0,TripA01,17.355765
1,TripA02,12.481613
2,TripA03,15.243973
3,TripA04,17.710491
4,TripA05,12.136815
5,TripA06,10.920328
6,TripA07,14.774177
7,TripA08,13.437999
8,TripA09,12.287638
9,TripA10,12.434787
